# Auxiliar 2 - Sonidos y su representación en el dominio de la frecuencia

## Reproducción de sonidos almacenados digitalmente

Los conversores digitales analógicos permiten recuperar una señal análógica (por ejemplo, un sonido) a partir de las muestras almacenadas en una serie de tiempo. El conversor sostendrá cada nivel en el período entre muestras aproximando a la señal original con una función constante a trozos (se puede sumar una etapa final de suavizado). 

Esta guía busca ejemplificar la relación entre tonos y frecuencias experimentando la relación entre el sonido y su representación en el diagrama de frecuencias.



### Bibliotecas de Python para reproducir sonidos

La biblioteca `simpleaudio` permite reproducir música almacenada en archivos .wav como, también, sonidos simples codificados en arreglos de `Numpy`. La idea es crear un objeto `WaveObject` que representará una pista de audio como un tren de valores, típicamente, codificados como enteros con signo de 16 bits (aunque admite codificaciones de 8, 16, 24 y 32 bits). Asimismo, se debe especificar el número de canales 1 (mon) o 2 (estéreo). La frecuencia de muestreo  permitidas (pero no necesariamente soportadas por el hardware) son: 8, 11.025, 16, 22.05, 32, 44.1, 48, 88.2, 96, and 192 kHz.

Dado que demostraremos la relación entre sonidos y representaciones dentro del formato de un *notebook*, aplicaremos la clase `Audio` del módulo `IPython.display`. La clase `Audio` permite reproducir sonidos desde diversas fuentes: arreglos de Numpy, listas, archivos, etc. La clase `Audio` requiere que la serie de tiempo tome valores entre -1 y 1. Sin embargo, si el parámetro `normalize` es `True`, la serie es escalada al rango permitido.   

A modo de ejemplo, en lo siguiente generamos 3 tonos, cada uno de duración 0,5 segundos y se toman muestras a una frecuencia de 44100Hz (44100 veces por segundo). 

In [ ]:
import numpy as np
import IPython.display as ipd

# calculate note frequencies
A_freq = 440
Csh_freq = A_freq * 2 ** (4 / 12)
E_freq = A_freq * 2 ** (7 / 12)

# get timesteps for each sample, T is note duration in seconds
sample_rate = 44100
T = 0.5
t = np.linspace(0, T, int(T * sample_rate))

# generate sine wave notes
A_note = np.sin(A_freq * t * 2 * np.pi)
Csh_note = np.sin(Csh_freq * t * 2 * np.pi)
E_note = np.sin(E_freq * t * 2 * np.pi)

# concatenate notes
audio = np.hstack((A_note, Csh_note, E_note))
# start playback
ipd.Audio(data=audio, rate=44100, autoplay=True)

### Reproducción de un ruido  

En audio, el concepto de ruido blanco que ocupa todas las frecuencias, se llama ruido rosa. En lo siguiente generamos una serie de valores aleatorios con distribución uniforme y escuchamos su sonido.

In [ ]:
# get timesteps for each sample, T is note duration in seconds
sample_rate = 44100
T = 1.5

# generate random values
mu, sigma = 0, 0.1 # mean and standard deviation
rng = np.random.default_rng()
pink_noise = mu + sigma * rng.standard_normal(int(T * sample_rate))

# start playback
ipd.Audio(data=pink_noise, rate=44100, autoplay=True)

### Fenómeno de aliasing

Vamos a probar de escuchar los sonidos producidos al elevar la frecuencia a partir de 2000Hz. Lo importante es notar que, al superar la frecuencia de Nyquist (en este caso 22050Hz), los sonidas escuchados se vuelven más graves aunque la frecuencia de la señal original continúa aumentando. Esto es consecuencia del efecto de *aliasing*. 

Por otro lado, notamos que la parte central de la pista de audio parece silenciarse. Sin embrago, ese sector corresponde a los tonos de frecuencias más elevados y suele ocurrir que nuestra audición solo percibe parcialmente los tonos de frecuencias mayores a 15000HZ (dependiendo de la salud y edad de la persona).


In [ ]:
import matplotlib.pyplot as plt

# get timesteps for each sample, T is note duration in seconds
sample_rate = 44100
T = 0.5
t = np.linspace(0, T, int(T * sample_rate))

audio = np.array([])
# calculate note frequencies
frequencies = np.linspace(2000,42000, 10)
for freq in frequencies: # barremos en frecuencia y excedemos a la frecuencia de Nyquist
    
    # generate sine wave notes
    tone = np.sin(freq * t * 2 * np.pi)
    audio = np.append(audio, tone)

times = T * np.arange(frequencies.size)
visual_freqs = np.hstack([frequencies.reshape(-1,1),frequencies.reshape(-1,1)]).flatten()
visual_times = np.hstack([times.reshape(-1,1),(times+T).reshape(-1,1)]).flatten()
plt.plot(visual_times, visual_freqs)
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("Frecuencias de audio generadas") 

# start playback
ipd.Audio(data=audio, rate=44100, autoplay=True)

## Relación entre sonidos y sus representaciones

Buscamos mostrar la relación entre sonidos y sus representaciones en el dominio del tiempo y en el dominio de la frecuencia. Queremos explorar cómo la experiencia de distintos sonidos se corresponden con las frecuencias y amplitudes de los armónicos que los componen. La transformada discreta de Fourier (DFT) es el método que perite hallar las amplitudes para las distintas frecuencias que componen la serie de tiempo original.

En lo siguiente, armamos un widget para variar la amplitud relativa entre dos tonos y cambiar la frecuencia de uno de ellos. Además, permitiremos modificar la frecuencia del segundo. Esperamos que al ver la representación y escuchar simultánemaente el tono, se comprenda mejor el trabajo que realiza la DFT para descomponer la serie de tiempo en suma de funciones armónicas. 



In [ ]:
from ipywidgets import interactive
import matplotlib.pyplot as plt

# get timesteps for each sample, T is note duration in seconds
sample_rate = 44100
T = 0.2
t = np.linspace(0, T, int(T * sample_rate))
freq1 = 440

# Define the function to update the plot
def update_plot(fade, freq2):

    # Señal
    audio1 = (1 - fade) * np.sin(2 * np.pi * t * freq1)
    audio2 = fade * np.sin(2 * np.pi * t * freq2)
    audio_t = audio1 + audio2 

    fig, ax = plt.subplots(1,2, figsize=(8,4))

    # Time domain
    ax[0].plot(t[:500], audio_t[:500],  label='Señal')
    ax[0].set_xlabel('Segundos')
    ax[0].set_ylabel('Amplitud')
    ax[0].set_title('Dominio del tiempo')
    ax[0].legend(loc='upper right')

    # Frequency domain
    audio_f = np.fft.rfft(audio_t)
    freqs = sample_rate * np.linspace(1/audio_t.size, 1/2, audio_t.size//2 if audio_t.size % 2 else 1 + audio_t.size//2)

    # Power spectrum
    ax[1].set_xscale('log')
    ax[1].stem(freqs, np.real(audio_f)**2 + np.imag(audio_f)**2, 'b', markerfmt='o', basefmt='b', label='Power')
    ax[1].set_xlim(100,10000)
    ax[1].set_xlabel('Frecuencia (Hz)')
    ax[1].set_title('Periodograma')
    ax[1].legend(loc='upper left')
    
    fig.tight_layout() 
    # plt.legend()
    # plt.grid(True)
    plt.show()

    # start playback
    ipd.display(ipd.Audio(data=audio_t, rate=44100, autoplay=True, normalize=True))
    
    
# Create interactive widgets
interactive_plot = interactive(update_plot, fade=(0, 1, 0.1), freq2=(500, 4000, 500))
output = interactive_plot.children[-1]
output.layout.height = '450px'
interactive_plot

### Ejercicios

1. Sumar un ruido rosa a los tonos anteriores y analizar los resultados. Explorar distintos tamaños para la varianza del ruido. ¿Cómo se podría eliminar el ruido para reobtener la señal filtrada?
2. Reemplazar el tono de 440Hz por una señal cuadrada de igual frecuencia. Explicar la presencia de más armónicos. (Hint: usar la función `np.sign()` o el generador de la biblioteca `scipy.signal.square`)
3. Retomar la función seno para el tono de 440Hz y cambiar la de frecuencia regulable por una señal cuadrada. ¿Podría manifestarse el fenómeno de *aliasing*? ¿En qué casos es más notable?
4. Modular la amplitud del tono de 440Hz por una señal senoidal de frecuencia variable $f$ entre 5Hz y 40Hz, y una amplitud $A$ ajustables de 0 a 1. Notar la presencia de bandas laterales.

$$ \sin(2 \pi t \,440\mathrm{Hz})  \left(1 + A \sin(2 \pi f t)\right) $$ 

4. La modulación por desplazamiento de frecuencia (FSK) es un esquema de modulación de frecuencia en el que la información digital se codifica en una señal portadora mediante el desplazamiento periódico de la frecuencia de la portadora entre varias frecuencias discretas. Modular la frecuencia del tono de 440Hz por una señal cuadrada de frecuencia variable $f$ entre 5Hz y 40Hz. Evaluar los resultados incrementando la frecuencia de la portadora entre un 10% y un 100% para los cambios de estado entre 0 y 1. ¿Es este tipo de modulación sensible al ruido aditivo? 


**Conclusión**: La  DFT (o su versión rápida) siempre nos permitirá descomponer una serie de tiempo en suma de funciones armónicas. Es importante destacar que el espectro de potencias nos informará cuáles son las frecuencias más importantes ayudando a descubrir ciclos u otros efectos que se repiten. Asimismo, podemos actuar sobre la señal digital en el espacio de frecuencias para eliminar ruidos o aplicar transformaciones.    